# Problem 1: Defining random seed

In [1]:
# Set seeds for reproducibility
import torch
import numpy as np

seed = 7
torch.manual_seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # This forces deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True 
    torch.backends.cudnn.benchmark = False

# Problem 1: Next Character Prediction (Basic Sequence)
This cell defines a dynamic, universal character-level model capable of switching between RNN, LSTM, and GRU architectures. It prepares a single-paragraph text dataset and performs a grid search across different sequence lengths (10, 20, 30) to compare training loss, validation accuracy, execution time, and model complexity.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time
from sklearn.model_selection import train_test_split

# Check for CUDA support and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# The provided sequence text
text = (
    "Next character prediction is a fundamental task in the field of natural language processing (NLP) "
    "that involves predicting the next character in a sequence of text based on the characters that precede it. "
    "This task is essential for various applications, including text auto-completion, spell checking, and even "
    "in the development of sophisticated AI models capable of generating human-like text. "
    "At its core, next character prediction relies on statistical models or deep learning algorithms to analyze "
    "a given sequence of text and predict which character is most likely to follow. These predictions are based "
    "on patterns and relationships learned from large datasets of text during the training phase of the model. "
    "One of the most popular approaches to next character prediction involves the use of Recurrent Neural Networks "
    "(RNNs), and more specifically, a variant called Long Short-Term Memory (LSTM) networks. RNNs are particularly "
    "well-suited for sequential data like text, as they can maintain information in 'memory' about previous "
    "characters to inform the prediction of the next character. LSTM networks enhance this capability by being "
    "able to remember long-term dependencies, making them even more effective for next character prediction tasks. "
    "Training a model for next character prediction involves feeding it large amounts of text data, allowing it "
    "to learn the probability of each character's appearance following a sequence of characters. During this "
    "training process, the model adjusts its parameters to minimize the difference between its predictions and "
    "the actual outcomes, thus improving its predictive accuracy over time. "
    "Once trained, the model can be used to predict the next character in a given piece of text by considering "
    "the sequence of characters that precede it. This can enhance user experience in text editing software, "
    "improve efficiency in coding environments with auto-completion features, and enable more natural interactions "
    "with AI-based chatbots and virtual assistants. "
    "In summary, next character prediction plays a crucial role in enhancing the capabilities of various NLP "
    "applications, making text-based interactions more efficient, accurate, and human-like. Through the use of "
    "advanced machine learning models like RNNs and LSTMs, next character prediction continues to evolve, "
    "opening new possibilities for the future of text-based technology."
)

# ---------------------------------------------------------
# 1. Dynamic Universal Model Definition
# ---------------------------------------------------------
class UniversalCharModel(nn.Module):
    def __init__(self, model_type, input_size, hidden_size, output_size):
        super(UniversalCharModel, self).__init__()
        self.hidden_size = hidden_size
        self.model_type = model_type
        
        # Embedding layer
        self.embedding = nn.Embedding(input_size, hidden_size)
        
        # Select RNN type based on parameter
        if self.model_type == 'RNN':
            self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        elif self.model_type == 'LSTM':
            self.rnn = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        elif self.model_type == 'GRU':
            self.rnn = nn.GRU(hidden_size, hidden_size, batch_first=True)
            
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        # Select the output of the last time step
        output = self.fc(output[:, -1, :]) 
        return output

# Helper function to count model parameters (Model Size Complexity)
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Helper function to prep data for a specific sequence length
def prepare_dataset(text, seq_length):
    chars = sorted(list(set(text)))
    char_to_ix = {ch: i for i, ch in enumerate(chars)}
    ix_to_char = {i: ch for i, ch in enumerate(chars)}
    
    X = []
    y = []
    for i in range(len(text) - seq_length):
        sequence = text[i:i + seq_length]
        label = text[i + seq_length]
        X.append([char_to_ix[char] for char in sequence])
        y.append(char_to_ix[label])

    X = np.array(X)
    y = np.array(y)

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    return (torch.tensor(X_train, dtype=torch.long).to(device),
            torch.tensor(X_val, dtype=torch.long).to(device),
            torch.tensor(y_train, dtype=torch.long).to(device),
            torch.tensor(y_val, dtype=torch.long).to(device),
            len(chars))

# ---------------------------------------------------------
# 2. Main Training Loop / Grid Search
# ---------------------------------------------------------
# Hyperparameters
hidden_size = 128
learning_rate = 0.005
epochs = 100

models_to_test = ['RNN', 'LSTM', 'GRU']
seq_lengths_to_test = [10, 20, 30]

# Dictionary to store all results for your report
results = {}

for seq_len in seq_lengths_to_test:
    print(f"==================================================")
    print(f"PREPARING DATA FOR SEQUENCE LENGTH: {seq_len}")
    print(f"==================================================")
    X_train, X_val, y_train, y_val, vocab_size = prepare_dataset(text, seq_len)
    
    for mod_type in models_to_test:
        print(f"\n--- Training {mod_type} with Sequence Length {seq_len} ---")
        
        # Instantiate model, loss, and optimizer
        model = UniversalCharModel(mod_type, vocab_size, hidden_size, vocab_size).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        
        # Record Model Size Complexity
        num_params = count_parameters(model)
        
        # Track time
        start_time = time.time()
        
        final_train_loss = 0.0
        final_val_acc = 0.0
        
        # Training loop
        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()
            output = model(X_train)
            loss = criterion(output, y_train)
            loss.backward()
            optimizer.step()
            
            # Validation
            model.eval()
            with torch.no_grad():
                val_output = model(X_val)
                val_loss = criterion(val_output, y_val)
                _, predicted = torch.max(val_output, 1)
                val_accuracy = (predicted == y_val).float().mean()
            
            if (epoch + 1) == epochs:  # Store final metrics
                final_train_loss = loss.item()
                final_val_acc = val_accuracy.item()
                
            if (epoch + 1) % 50 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | Val Acc: {val_accuracy.item():.4f}")
                
        end_time = time.time()
        execution_time = end_time - start_time
        
        # Save results for final summary
        key = f"{mod_type}_Seq{seq_len}"
        results[key] = {
            'Model': mod_type,
            'Sequence Length': seq_len,
            'Train Loss': final_train_loss,
            'Validation Accuracy': final_val_acc,
            'Execution Time (s)': execution_time,
            'Model Size (Params)': num_params
        }

# ---------------------------------------------------------
# 3. Print Summary Table for the Report
# ---------------------------------------------------------
print("\n" + "="*85)
print(f"{'Model':<10} | {'Seq Len':<8} | {'Train Loss':<12} | {'Val Acc':<10} | {'Exec Time (s)':<15} | {'Params':<10}")
print("="*85)
for key, res in results.items():
    print(f"{res['Model']:<10} | {res['Sequence Length']:<8} | {res['Train Loss']:<12.4f} | {res['Validation Accuracy']:<10.4f} | {res['Execution Time (s)']:<15.4f} | {res['Model Size (Params)']:<10}")
print("="*85)

Using device: cuda

PREPARING DATA FOR SEQUENCE LENGTH: 10

--- Training RNN with Sequence Length 10 ---
Epoch 50/100 | Loss: 0.7927 | Val Acc: 0.5315
Epoch 100/100 | Loss: 0.0843 | Val Acc: 0.4937

--- Training LSTM with Sequence Length 10 ---
Epoch 50/100 | Loss: 1.0329 | Val Acc: 0.5189
Epoch 100/100 | Loss: 0.1247 | Val Acc: 0.5105

--- Training GRU with Sequence Length 10 ---
Epoch 50/100 | Loss: 0.7474 | Val Acc: 0.5357
Epoch 100/100 | Loss: 0.0645 | Val Acc: 0.5105
PREPARING DATA FOR SEQUENCE LENGTH: 20

--- Training RNN with Sequence Length 20 ---
Epoch 50/100 | Loss: 0.8100 | Val Acc: 0.5000
Epoch 100/100 | Loss: 0.0839 | Val Acc: 0.5021

--- Training LSTM with Sequence Length 20 ---
Epoch 50/100 | Loss: 0.9624 | Val Acc: 0.5127
Epoch 100/100 | Loss: 0.1314 | Val Acc: 0.4831

--- Training GRU with Sequence Length 20 ---
Epoch 50/100 | Loss: 0.7600 | Val Acc: 0.5401
Epoch 100/100 | Loss: 0.0575 | Val Acc: 0.5380
PREPARING DATA FOR SEQUENCE LENGTH: 30

--- Training RNN with Sequ

## Problem 1: Inference & Text Generation
This cell evaluates the qualitative performance of the final trained model from the previous loop. It uses a seed string to predict and generate the next 100 characters in the sequence, demonstrating the model's learned patterns and limitations.

In [3]:
# ---------------------------------------------------------
# 4. Text Generation / Prediction
# ---------------------------------------------------------
def generate_text(model, seed_text, generate_length, seq_length, text_data):
    """
    Generates new text using a trained model.
    """
    model.eval()
    
    # Recreate the vocab mappings based on the original text
    chars = sorted(list(set(text_data)))
    char_to_ix = {ch: i for i, ch in enumerate(chars)}
    ix_to_char = {i: ch for i, ch in enumerate(chars)}
    
    generated = seed_text
    
    # Pad or truncate the seed text to exactly match the sequence length
    if len(seed_text) < seq_length:
        # If seed is too short, pad it with spaces (or handle differently based on preference)
        current_seq = seed_text.rjust(seq_length, ' ')
    else:
        current_seq = seed_text[-seq_length:]
        
    with torch.no_grad():
        for _ in range(generate_length):
            # Convert current sequence to tensor
            input_seq = torch.tensor([char_to_ix[c] for c in current_seq], dtype=torch.long).unsqueeze(0).to(device)
            
            # Get prediction
            prediction = model(input_seq)
            predicted_index = torch.argmax(prediction, dim=1).item()
            predicted_char = ix_to_char[predicted_index]
            
            # Append predicted character to the generated text
            generated += predicted_char
            
            # Update the current sequence by sliding the window forward
            current_seq = current_seq[1:] + predicted_char
            
    return generated

# Example usage: Test generation using the very last trained model in the loop
# (which would be GRU with sequence length 30 based on our previous loop)
print("\n" + "="*85)
print("TESTING TEXT GENERATION")
print("="*85)

test_seed = "Next character prediction "
# Make sure to use the seq_len variable from the last iteration of your training loop
gen_text = generate_text(model, test_seed, generate_length=100, seq_length=seq_len, text_data=text)

print(f"Seed Text: '{test_seed}'")
print(f"Generated Text:\n{gen_text}")
print("="*85)


TESTING TEXT GENERATION
Seed Text: 'Next character prediction '
Generated Text:
Next character prediction involves predicting the next character in a sequence of text and predictions and the a fundamental t
